# Face Age — MRI-Derived Facial Morphometrics for Age Regression

**Research question:** Do MRI-derived face age and brain age from the same scan capture the same or independent biological aging signals?

**Pipeline overview:**
```
T1 MRI (NIfTI)
    │
    ▼  extract_head_mesh()          isosurface + PyMeshLab cleanup
Full head mesh (.ply)
    │
    ▼  center_and_extract_face()    crop + RAS→LSA reorientation
Face mesh (.ply, LSA space)
    │
    ▼  detect_landmarks()           BioFace3D-20 MVCNN (Deep-MVLM)
20 facial landmarks  (20 × 3 mm)
    │
    ├──▶  GPA features    Procrustes alignment + PCA  → ≤50 PC scores
    └──▶  EDMA features   190 pairwise distances (mm)
                                    │
                                    ▼
                              MLP age regressor
```

**Dataset:** IXI (Information eXtraction from Images) — 563 healthy subjects, 3 UK sites, age 20–86 yr  
**Reference:** Heredia-Lidón et al., *Computer Methods and Programs in Biomedicine*, 2025 (BioFace3D)

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import pyvista as pv

ROOT     = Path("..")
DATA     = ROOT / "data"
FEAT     = DATA / "features"
MESHES   = ROOT / "results" / "ixi_meshes"
MODELS   = DATA / "models"

plt.rcParams.update({"figure.dpi": 120, "font.size": 11})

---
## 1. Dataset Statistics

In [ ]:
meta  = pd.read_csv(DATA / "ixi_metadata.csv")
train = pd.read_csv(DATA / "ixi_train.csv")
val   = pd.read_csv(DATA / "ixi_val.csv")
test  = pd.read_csv(DATA / "ixi_test.csv")

# ── Overall summary table ──────────────────────────────────────────────────
def split_summary(df, name):
    return {
        "Split":    name,
        "N":        len(df),
        "Age mean": f"{df.age.mean():.1f}",
        "Age SD":   f"{df.age.std():.1f}",
        "Age min":  f"{df.age.min():.0f}",
        "Age max":  f"{df.age.max():.0f}",
        "Male":     f"{(df.sex=='M').sum()}  ({100*(df.sex=='M').mean():.0f}%)",
        "Female":   f"{(df.sex=='F').sum()}  ({100*(df.sex=='F').mean():.0f}%)",
    }

summary = pd.DataFrame([split_summary(meta, "All"),
                         split_summary(train, "Train"),
                         split_summary(val,   "Val"),
                         split_summary(test,  "Test")]).set_index("Split")
display(summary)

In [ ]:
# ── Site × Sex breakdown ───────────────────────────────────────────────────
site_sex = meta.groupby(["site", "sex"]).size().unstack(fill_value=0)
site_sex["Total"] = site_sex.sum(axis=1)
display(site_sex)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Age distribution
ax = axes[0]
for df, label, color in [(train, "Train", "steelblue"), (val, "Val", "orange"), (test, "Test", "green")]:
    ax.hist(df.age, bins=15, alpha=0.6, label=label, color=color, edgecolor="white")
ax.set_xlabel("Age (years)")
ax.set_ylabel("Count")
ax.set_title("Age Distribution by Split")
ax.legend()

# Age by site
ax = axes[1]
sites = sorted(meta.site.unique())
ax.boxplot([meta.loc[meta.site == s, "age"].values for s in sites],
           labels=sites, patch_artist=True,
           boxprops=dict(facecolor="lightsteelblue"),
           medianprops=dict(color="navy", linewidth=2))
ax.set_xlabel("Site")
ax.set_ylabel("Age (years)")
ax.set_title("Age Distribution by Site")

# Age by sex
ax = axes[2]
ax.boxplot([meta.loc[meta.sex == s, "age"].values for s in ["M", "F"]],
           labels=["Male", "Female"], patch_artist=True,
           boxprops=dict(facecolor="lightcoral"),
           medianprops=dict(color="darkred", linewidth=2))
ax.set_xlabel("Sex")
ax.set_ylabel("Age (years)")
ax.set_title("Age Distribution by Sex")

plt.tight_layout()
plt.show()

**Key observations:**
- 563 usable subjects after dropping 18 with no metadata (all 606 files scanned; 43 T1s are not in the IXI spreadsheet)
- Age range 20–86 yr, roughly uniform; slight female majority (55.6%)
- Guys dominates (55.8%) — site effects are a known confound in brain/face age studies
- All three splits have matched age means and M/F ratios (stratified by site × sex × age-decade)

---
## 2. Feature Extraction

### Prepare metadata and splits

Parses subject IDs and sites from T1 filenames, merges with `IXI.xls`, drops subjects with missing age, and writes a frozen 70/15/15 split stratified by site × sex × age-decade.  
The split is **locked** — re-running without `--force` is a no-op.

In [ ]:
!python ../scripts/prepare_metadata.py

### Run the full feature extraction pipeline

For each subject in all three splits:
1. **Head mesh** — isosurface (VTK FlyingEdges3D) on histogram-matched T1, cleaned with PyMeshLab
2. **Face mesh** — bounding-box centering, posterior/inferior crop, reorient to LSA space (required by BioFace3D)
3. **Landmarks** — BioFace3D-20 MVCNN (Deep-MVLM, Paulsen et al.) — 20 anatomical 3D points, MRE ≈ 1.61 mm on IXI

Then:
4. **GPA** — fit Procrustes + PCA on **train only**; transform val/test using train mean shape
5. **EDMA** — 190 pairwise distances per subject (no fitting required)

The script is fully **resumable** — already-extracted subjects are skipped automatically.

> Expected runtime: ~2 min per subject on Apple Silicon MPS → ~18 h for all 563 subjects

In [ ]:
# Run extraction (or just reassemble features from existing landmarks)
!python ../scripts/extract_features.py --device mps --predict-num 5

# Once all landmarks exist, features-only mode is instant:
# !python ../scripts/extract_features.py --features-only

### Biomarker descriptions

#### GPA — Generalized Procrustes Analysis
All 20-landmark configurations are iteratively aligned (translate → scale to unit centroid size → SVD rotation) until the mean shape converges. PCA is then applied to the flattened aligned coordinates. Each subject's **PC scores** (≤50 values) describe where their face shape sits in the population morphospace — capturing variation in face width, elongation, symmetry, etc. — with position, size, and orientation removed.

GPA is a **population-level** operation: val/test subjects are projected through the train mean shape and PCA without refitting.

#### EDMA — Euclidean Distance Matrix Analysis
For each subject, compute the 20×20 symmetric matrix of all pairwise Euclidean distances between landmarks. The 190 unique upper-triangle values form the EDMA feature vector. Each value is simply the straight-line distance (mm) between landmark pair (*i*, *j*). Unlike GPA, EDMA is computed **independently per subject** — no population fitting — and retains absolute size information.

---
## 3. 3D Face Visualisation with Landmarks

Three subjects from the dataset. Face meshes are shown in LSA space (Left-Superior-Anterior), which is the coordinate system the BioFace3D model was trained in. Red spheres = 20 detected landmarks.

In [ ]:
SAMPLE_IDS = [2, 12, 14]   # IXI002, IXI012, IXI014

def load_sample(ixi_id):
    sid   = f"IXI{ixi_id:03d}"
    face  = MESHES / f"{sid}_face.ply"
    lm_np = MESHES / f"{sid}_landmarks.npy"
    if not face.exists() or not lm_np.exists():
        return None, None, None
    mesh = pv.read(str(face))
    lm   = np.load(str(lm_np))
    row  = meta[meta.subject_id == ixi_id].iloc[0]
    return mesh, lm, row

def mesh_to_plotly(mesh, lm, label):
    pts   = mesh.points
    faces = mesh.faces.reshape(-1, 4)[:, 1:]   # strip VTK leading count
    surf = go.Mesh3d(
        x=pts[:, 0], y=pts[:, 1], z=pts[:, 2],
        i=faces[:, 0], j=faces[:, 1], k=faces[:, 2],
        color="#d4a58a", opacity=0.75, flatshading=False,
        lighting=dict(diffuse=0.8, specular=0.3, ambient=0.4),
        name="Face mesh", showlegend=False,
    )
    points = go.Scatter3d(
        x=lm[:, 0], y=lm[:, 1], z=lm[:, 2],
        mode="markers+text",
        text=[str(i) for i in range(len(lm))],
        textposition="top center",
        textfont=dict(size=8, color="white"),
        marker=dict(size=5, color="crimson", symbol="circle"),
        name="Landmarks",
    )
    return surf, points


fig = make_subplots(
    rows=1, cols=3,
    specs=[[{"type": "scene"}, {"type": "scene"}, {"type": "scene"}]],
    subplot_titles=[f"IXI{sid:03d}" for sid in SAMPLE_IDS],
)

for col, ixi_id in enumerate(SAMPLE_IDS, 1):
    mesh, lm, row = load_sample(ixi_id)
    if mesh is None:
        print(f"IXI{ixi_id:03d}: not yet extracted — run extract_features.py first")
        continue
    surf, pts_trace = mesh_to_plotly(mesh, lm, f"IXI{ixi_id:03d}")
    fig.add_trace(surf,       row=1, col=col)
    fig.add_trace(pts_trace,  row=1, col=col)
    fig.layout.annotations[col - 1].text = (
        f"IXI{ixi_id:03d} · {row.sex} · {row.age:.0f} yr · {row.site}"
    )

camera = dict(eye=dict(x=0, y=-2.2, z=0.5))
for i in range(1, 4):
    scene_key = f"scene{'' if i == 1 else i}"
    fig.update_layout(**{
        scene_key: dict(
            camera=camera,
            xaxis=dict(visible=False),
            yaxis=dict(visible=False),
            zaxis=dict(visible=False),
            bgcolor="#1a1a2e",
        )
    })

fig.update_layout(
    height=500, width=1100,
    title="Face meshes with BioFace3D-20 landmarks (LSA coordinate space)",
    paper_bgcolor="#1a1a2e", font_color="white",
    margin=dict(t=60, b=10, l=10, r=10),
)
fig.show()

---
## 4. Biomarker Visualisation

### 4a. GPA Morphospace

In [ ]:
feat_path = FEAT / "train.npz"
if not feat_path.exists():
    print("train.npz not yet available — run extract_features.py --features-only after landmarks are done.")
else:
    d = np.load(str(feat_path), allow_pickle=True)
    gpa  = d["gpa_scores"]
    ages = d["ages"]
    sexes = d["sexes"]
    sites = d["sites"]

    fig, axes = plt.subplots(1, 2, figsize=(13, 5))

    # PC1 vs PC2 coloured by age
    sc = axes[0].scatter(gpa[:, 0], gpa[:, 1], c=ages, cmap="RdYlBu_r",
                         alpha=0.7, s=30, edgecolors="none")
    plt.colorbar(sc, ax=axes[0], label="Age (years)")
    axes[0].set_xlabel("PC1")
    axes[0].set_ylabel("PC2")
    axes[0].set_title("GPA Morphospace — coloured by age")

    # PC1 vs PC2 coloured by site
    site_colors = {s: c for s, c in zip(np.unique(sites), ["steelblue", "tomato", "seagreen"])}
    for site in np.unique(sites):
        mask = sites == site
        axes[1].scatter(gpa[mask, 0], gpa[mask, 1], label=site, alpha=0.6,
                        s=30, color=site_colors[site], edgecolors="none")
    axes[1].set_xlabel("PC1")
    axes[1].set_ylabel("PC2")
    axes[1].set_title("GPA Morphospace — coloured by site")
    axes[1].legend()

    plt.tight_layout()
    plt.show()

    # Variance explained
    import joblib
    pca_model = joblib.load(str(FEAT / "gpa_pca.joblib"))
    fig, ax = plt.subplots(figsize=(8, 3))
    ve = pca_model.explained_variance_ratio_
    ax.bar(range(1, len(ve) + 1), ve * 100, color="steelblue", alpha=0.8)
    ax.plot(range(1, len(ve) + 1), np.cumsum(ve) * 100, "o-", color="tomato",
            markersize=3, label="Cumulative")
    ax.axhline(80, color="grey", linestyle="--", linewidth=0.8, label="80% threshold")
    ax.set_xlabel("PC")
    ax.set_ylabel("Variance explained (%)")
    ax.set_title("GPA PCA — Scree Plot")
    ax.legend()
    plt.tight_layout()
    plt.show()

### 4b. EDMA — Mean Inter-Landmark Distances

In [ ]:
if not feat_path.exists():
    print("train.npz not yet available.")
else:
    edma = d["edma_distances"]   # (N, 190)
    N_LM = 20
    triu_i, triu_j = np.triu_indices(N_LM, k=1)

    # Rebuild mean form matrix
    mean_fm = np.zeros((N_LM, N_LM))
    mean_fm[triu_i, triu_j] = edma.mean(axis=0)
    mean_fm += mean_fm.T

    fig, ax = plt.subplots(figsize=(7, 6))
    im = ax.imshow(mean_fm, cmap="viridis", aspect="auto")
    plt.colorbar(im, ax=ax, label="Mean distance (mm)")
    ax.set_title("EDMA — Mean inter-landmark distance matrix (train set)")
    ax.set_xlabel("Landmark index")
    ax.set_ylabel("Landmark index")
    plt.tight_layout()
    plt.show()

### 4c. Per-subject biomarker table — 3 samples

In [ ]:
import sys; sys.path.insert(0, str(ROOT))
from src.biomarkers import edma_form_matrix

N_LM = 20
triu_i, triu_j = np.triu_indices(N_LM, k=1)

# Top-15 longest distances for each sample (most informative pairs)
rows = []
for ixi_id in SAMPLE_IDS:
    lm_path = MESHES / f"IXI{ixi_id:03d}_landmarks.npy"
    if not lm_path.exists():
        continue
    lm  = np.load(str(lm_path))
    row = meta[meta.subject_id == ixi_id].iloc[0]
    fm  = edma_form_matrix(lm)
    dists = fm[triu_i, triu_j]
    idx_sorted = np.argsort(dists)[::-1][:15]
    for rank, k in enumerate(idx_sorted, 1):
        rows.append({
            "Subject":  f"IXI{ixi_id:03d}",
            "Sex":      row.sex,
            "Age (yr)": f"{row.age:.0f}",
            "Rank":     rank,
            "LM pair":  f"({triu_i[k]}, {triu_j[k]})",
            "Distance (mm)": f"{dists[k]:.1f}",
        })

table = pd.DataFrame(rows)
display(
    table.pivot_table(
        index=["Subject", "Sex", "Age (yr)", "Rank"],
        values=["LM pair", "Distance (mm)"],
        aggfunc="first",
    ).head(45).style.set_caption("Top-15 longest inter-landmark distances per subject")
)

In [ ]:
# Centroid sizes (head size proxy) from GPA output
if feat_path.exists():
    cs_all  = d["centroid_sizes"]
    age_all = d["ages"]

    fig, ax = plt.subplots(figsize=(7, 4))
    ax.scatter(age_all, cs_all, alpha=0.5, s=20, color="steelblue", edgecolors="none")
    m, b = np.polyfit(age_all, cs_all, 1)
    x_line = np.linspace(age_all.min(), age_all.max(), 100)
    ax.plot(x_line, m * x_line + b, "--", color="tomato", linewidth=1.5, label=f"slope={m:.3f}")
    ax.set_xlabel("Age (years)")
    ax.set_ylabel("Centroid size (a.u.)")
    ax.set_title("Centroid size (head size proxy) vs age — train set")
    ax.legend()
    plt.tight_layout()
    plt.show()

---
## 5. Age Regression — MLP Model

### Model architecture

A fully-connected MLP trained end-to-end on morphometric features to predict chronological age.

| Component | Choice | Rationale |
|---|---|---|
| Input | GPA (≤50) + EDMA (190) = ≤240 dims | Complementary: shape + size |
| Hidden layers | 256 → 128 → 64 | Sufficient capacity for <600 subjects |
| Activations | ReLU + BatchNorm + Dropout (0.3) | Regularisation given small N |
| Output | Linear (scalar) | Regression |
| Loss | MSE | Standard for regression |
| Optimiser | Adam, lr=1e-3, weight decay=1e-4 | |
| LR schedule | ReduceLROnPlateau (×0.5 when val MAE plateaus) | |
| Early stopping | Patience=30 epochs on val MAE | Prevents overfitting on N=402 train |
| Feature scaling | StandardScaler (fit on train only) | |

**Metrics reported:** MAE (primary), RMSE, Pearson R, R²  
**Breakdowns:** per-site (Guys / HH / IOP), per-sex (M / F)  
**Baseline to beat:** predicting mean age → MAE ≈ 13 yr on this dataset

Three runs compare feature sets:
- `gpa` — shape only (50 PCs)
- `edma` — distances only (190 values)
- `both` — combined (≤240 values)

In [ ]:
# Train with combined features (default)
!python ../scripts/train_age_regressor.py --features both --device auto

In [ ]:
# Compare all three feature sets
!python ../scripts/train_age_regressor.py --features gpa  --device auto --run-name gpa_256-128-64
!python ../scripts/train_age_regressor.py --features edma --device auto --run-name edma_256-128-64

### Results

In [ ]:
def load_run(run_name):
    run_dir = MODELS / run_name
    if not run_dir.exists():
        return None
    rows = []
    for split in ("train", "val", "test"):
        p = run_dir / f"{split}_predictions.csv"
        if p.exists():
            df = pd.read_csv(p)
            df["split"] = split
            df["run"]   = run_name
            rows.append(df)
    return pd.concat(rows, ignore_index=True) if rows else None

runs = {
    "both":  load_run("both_256-128-64"),
    "gpa":   load_run("gpa_256-128-64"),
    "edma":  load_run("edma_256-128-64"),
}
runs = {k: v for k, v in runs.items() if v is not None}

if not runs:
    print("No model runs found — train the models first using the cells above.")
else:
    # Summary table: MAE / RMSE / R² per run × split
    summary_rows = []
    for run_name, df in runs.items():
        for split in df.split.unique():
            sub = df[df.split == split]
            mae  = sub.abs_error.mean()
            rmse = np.sqrt((sub.error**2).mean())
            r2   = 1 - ((sub.error**2).sum() / ((sub.age_true - sub.age_true.mean())**2).sum())
            summary_rows.append({"Features": run_name, "Split": split,
                                  "MAE (yr)": round(mae, 2),
                                  "RMSE (yr)": round(rmse, 2),
                                  "R²": round(r2, 3)})
    display(
        pd.DataFrame(summary_rows)
          .pivot(index="Features", columns="Split", values=["MAE (yr)", "RMSE (yr)", "R²"])
          .style.set_caption("Age regression results")
          .format("{:.2f}")
    )

---
## 6. Model Comparison — Benchmark

Six regressors trained on the same features (GPA + EDMA), all hyperparameters tuned by
5-fold cross-validation on the **train set only**.

| Model | Description | Key hyperparameters |
|---|---|---|
| **Ridge** | Linear + L2 penalty | α (RidgeCV, log-spaced grid) |
| **Lasso** | Linear + L1 penalty — sparse solution | α (LassoCV) |
| **ElasticNet** | Linear + L1+L2 — best of both | α, l1_ratio (ElasticNetCV) |
| **SVR** | Support Vector Regression, RBF kernel | C, ε (GridSearchCV 5-fold) |
| **GBM** | Gradient Boosting (sklearn) | n_estimators, max_depth, lr, subsample |
| **MLP small** | 2-layer MLP 64→32, early stopping | α=1e-3, adaptive lr |

All models use the same `StandardScaler` fitted on train features.

In [ ]:
# Run the full benchmark
# The script is resume-safe: already-fitted models are skipped automatically.
# If the session was interrupted after the linear models, run only the remaining three:
#   !python ../scripts/benchmark_regressors.py --models svr gbm mlp
# For a clean run from scratch:
!python ../scripts/benchmark_regressors.py --features both

In [ ]:
BENCH = ROOT / "data" / "models" / "benchmark"
results = pd.read_csv(BENCH / "benchmark_results.csv")

display(
    results.sort_values("test_mae")
    [[ "model","train_mae","val_mae","test_mae",
       "train_rmse","val_rmse","test_rmse",
       "test_r2","test_r","test_bias","train_time_s"]]
    .rename(columns={
        "model":"Model", "train_mae":"Train MAE", "val_mae":"Val MAE",
        "test_mae":"Test MAE", "train_rmse":"Train RMSE", "val_rmse":"Val RMSE",
        "test_rmse":"Test RMSE", "test_r2":"Test R²", "test_r":"Test r",
        "test_bias":"Bias", "train_time_s":"Time (s)"
    })
    .set_index("Model")
    .style
    .format("{:.2f}")
    .background_gradient(subset=["Test MAE"], cmap="RdYlGn_r")
    .set_caption("Benchmark results — sorted by test MAE (lower is better)")
)

In [ ]:
# Overfitting gap: train MAE vs val/test MAE
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

models_ordered = results.sort_values("test_mae")["model"].tolist()
x = np.arange(len(models_ordered))
w = 0.28

# MAE comparison
ax = axes[0]
ax.bar(x - w, results.set_index("model").loc[models_ordered, "train_mae"], w, label="Train", color="steelblue", alpha=0.85)
ax.bar(x,     results.set_index("model").loc[models_ordered, "val_mae"],   w, label="Val",   color="orange",    alpha=0.85)
ax.bar(x + w, results.set_index("model").loc[models_ordered, "test_mae"],  w, label="Test",  color="tomato",    alpha=0.85)
ax.set_xticks(x); ax.set_xticklabels(models_ordered, rotation=20)
ax.set_ylabel("MAE (years)"); ax.set_title("MAE by split")
ax.axhline(results.set_index("model").loc["ridge","test_mae"], color="grey",
           linestyle="--", linewidth=0.8, label="Ridge baseline")
ax.legend(fontsize=9)

# Test R²
ax = axes[1]
colors = ["steelblue" if r == results["test_mae"].min() else "lightsteelblue"
          for r in results.set_index("model").loc[models_ordered, "test_mae"]]
ax.bar(x, results.set_index("model").loc[models_ordered, "test_r2"], color=colors, alpha=0.85)
ax.set_xticks(x); ax.set_xticklabels(models_ordered, rotation=20)
ax.set_ylabel("R²"); ax.set_title("Test R²  (higher is better)")

# Overfitting gap
ax = axes[2]
gap = results.set_index("model").loc[models_ordered, "test_mae"] - \
      results.set_index("model").loc[models_ordered, "train_mae"]
bar_colors = ["tomato" if g > 3 else "seagreen" for g in gap]
ax.bar(x, gap.values, color=bar_colors, alpha=0.85)
ax.set_xticks(x); ax.set_xticklabels(models_ordered, rotation=20)
ax.set_ylabel("Test MAE − Train MAE (years)"); ax.set_title("Overfitting gap")
ax.axhline(0, color="black", linewidth=0.8)

plt.tight_layout()
plt.show()

### 6b. Full Results Table (all 6 models)

Recomputed from individual prediction CSVs so the table is always complete
regardless of whether the benchmark was run in one go or resumed.


In [ ]:
from scipy.stats import pearsonr

BENCH = ROOT / 'data' / 'models' / 'benchmark'
ALL_MODELS = ['ridge', 'lasso', 'elastic', 'svr', 'gbm', 'mlp']

rows = []
for name in ALL_MODELS:
    row = {'model': name}
    for split in ('train', 'val', 'test'):
        p = BENCH / f'{name}_{split}_predictions.csv'
        if not p.exists():
            continue
        df = pd.read_csv(p)
        mae  = df.abs_error.mean()
        rmse = np.sqrt((df.error**2).mean())
        r2   = 1 - (df.error**2).sum() / ((df.age_true - df.age_true.mean())**2).sum()
        r, _ = pearsonr(df.age_true, df.age_pred)
        bias = df.error.mean()
        row[f'{split}_mae']  = round(mae,  2)
        row[f'{split}_rmse'] = round(rmse, 2)
        row[f'{split}_r2']   = round(r2,   3)
        row[f'{split}_r']    = round(float(r), 3)
        row[f'{split}_bias'] = round(bias, 2)
    rows.append(row)

full_results = pd.DataFrame(rows)

display(
    full_results.sort_values('test_mae')
    [['model','train_mae','val_mae','test_mae',
      'train_r2','val_r2','test_r2','test_r','test_bias']]
    .rename(columns={
        'model':'Model',
        'train_mae':'Train MAE', 'val_mae':'Val MAE', 'test_mae':'Test MAE',
        'train_r2':'Train R²',  'val_r2':'Val R²',   'test_r2':'Test R²',
        'test_r':'Test r',      'test_bias':'Bias (yrs)'
    })
    .style.highlight_min(subset=['Test MAE'], color='lightgreen')
           .highlight_max(subset=['Test R²', 'Test r'], color='lightgreen')
           .format(precision=2)
)

### 6d. Model Selection — Verdict

#### Summary table

| Model | Test MAE | Test R² | Train-Test gap | Notes |
|---|---|---|---|---|
| **Ridge** | **8.31** | **0.538** | +0.24 | Best overall; near-zero bias (−0.29 yr); tied best r=0.739 |
| Lasso | 8.38 | 0.525 | +0.09 | Marginal sparsity benefit; slightly worse R² |
| ElasticNet | 8.47 | 0.511 | −0.20 | Slightly underfits; no advantage over Ridge |
| SVR | 8.86 | 0.532 | **+7.86** | Near-perfect train fit; severe overfit despite CV tuning |
| GBM | 8.90 | 0.483 | **+8.35** | Worst overfit; best train MAE (0.55) yet worst nonlinear test |
| MLP | 11.63 | 0.013 | +3.58 | Clearly underpowered for n=335; val R²≈0 |

#### Winner: **Ridge Regression**

**Why Ridge wins:**
- Lowest test MAE (8.31 yr) and highest test R² (0.538).
- Negligible overfitting gap (+0.24 yr) — the val MAE (8.65) reliably
  predicted the test MAE, confirming the model generalises.
- Near-zero prediction bias (−0.29 yr); the age scale is preserved.
- Consistent across both sites (Guys 8.14, HH 8.65) and sexes (M 7.19, F 9.26).

**Why the nonlinear models underperform:**
- The dataset is wide relative to its depth (n=335, d=240).
  In this regime, high-variance methods (SVR, GBM) overfit even with
  5-fold CV tuning because each fold's training set has only ~268 samples.
- The face morphometric → age mapping appears to be dominated by a
  monotone, near-linear trend (global shape/size change with age);
  nonlinear capacity does not add signal, only variance.

**Why MLP fails:**
- Even the small 64→32 architecture has ~16 k parameters for 335 samples.
  Early stopping prevents full memorisation, but the model never finds a
  stable solution: val R²≈0.03, test R²≈0.01.

**Sex effect (consistent across all models):**
Males are predicted ~2 yr more accurately than females (M ≈7 yr, F ≈9 yr MAE).
This likely reflects greater morphological variability in female faces
across the IXI age range, rather than a model artefact.

**Recommendation:** use **Ridge** as the face-age predictor for the
subsequent face-age vs brain-age comparison.
